# 04 - Evaluation: aggregate metrics, slice tables, RAG faithfulness, failure cases

Phase J. This notebook is the project's quantitative evidence layer. It deliberately *does not retrain*; it loads the artifacts produced in Phases C, D, F, and re-evaluates them on the held-out test split.

Sections:
1. Aggregate metrics for ML and DL on the held-out test set.
2. Per-slice tables (sex, age band) for both models.
3. Top-k highest-confidence wrong predictions (calibration smell test).
4. RAG faithfulness: do the LLM explanations cite only available evidence and share vocabulary with it?
5. Failure-mode summary for the synthesis paper.

In [ ]:
import sys, json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data_loader import load_heart_disease
from src.preprocessing import split_and_preprocess
from src.decision import load_default_scorers
from src.dl_model import predict_proba as dl_predict_proba
from src.ml_model import predict_proba as ml_predict_proba
from src.evaluation import (
    age_bucket,
    aggregate_metrics,
    failure_cases,
    rag_faithfulness,
    slice_table,
)
from src.rag.retriever import search
from src.agent_orchestrator import run as agent_run

In [ ]:
df = load_heart_disease()
split = split_and_preprocess(df)
ml, dl, pp = load_default_scorers(split)
ml_probs = ml_predict_proba(ml, split.X_test)
dl_probs = dl_predict_proba(dl, pp, split.X_test)
print('test n =', len(split.X_test), '| positive rate =', round(float(split.y_test.mean()), 3))

## 1. Aggregate metrics

In [ ]:
ml_m = aggregate_metrics(ml_probs, split.y_test)
dl_m = aggregate_metrics(dl_probs, split.y_test)
agg = pd.DataFrame([
    {'model': 'ML (HistGradientBoosting)', **ml_m.__dict__},
    {'model': 'DL (PyTorch MLP)', **dl_m.__dict__},
])
agg

## 2. Slice tables
Lineage: Project 4 disaggregated evaluation. Headline AUC can hide large per-subgroup gaps; we look at sex and age band for both models.

In [ ]:
sex_series = split.X_test['sex']
age_series = split.X_test['age'].apply(age_bucket).rename('age_band')

print('--- ML by sex ---')
print(slice_table(ml_probs, split.y_test, sex_series, label='sex').to_string(index=False))
print()
print('--- DL by sex ---')
print(slice_table(dl_probs, split.y_test, sex_series, label='sex').to_string(index=False))

In [ ]:
print('--- ML by age band ---')
print(slice_table(ml_probs, split.y_test, age_series, label='age_band').sort_values('age_band').to_string(index=False))
print()
print('--- DL by age band ---')
print(slice_table(dl_probs, split.y_test, age_series, label='age_band').sort_values('age_band').to_string(index=False))

## 3. Top failure cases
Highest-confidence wrong predictions (decision threshold 0.5). These are the rows a clinician would be most misled by.

In [ ]:
ml_fail = failure_cases(split.X_test, split.y_test, ml_probs, k=5)
dl_fail = failure_cases(split.X_test, split.y_test, dl_probs, k=5)
print('--- ML top-5 most-confident wrong ---')
print(ml_fail[['age', 'sex', 'cp', 'thal', 'oldpeak', 'ca', 'y_true', 'prob', 'pred']].to_string())
print()
print('--- DL top-5 most-confident wrong ---')
print(dl_fail[['age', 'sex', 'cp', 'thal', 'oldpeak', 'ca', 'y_true', 'prob', 'pred']].to_string())

## 4. RAG faithfulness
For each of three live runs we capture the explanation text and the retrieved evidence chunks, then check:
- every `[S?]` index in the explanation is within range of the retrieved chunks (no fabricated citations);
- the count of unique citations vs the number of chunks made available;
- token overlap between the explanation and the cited chunks (a crude grounding signal).

We re-run the agent on three test patients and inspect the *draft* event from the run log to grab the explanation + retrieved chunks together.

In [ ]:
# Pick three patients with varied risk
ml_series = pd.Series(ml_probs, index=split.X_test.index)
low_idx = ml_series.idxmin()
mid_idx = (ml_series - 0.5).abs().idxmin()
high_idx = ml_series.idxmax()
sample_indices = [high_idx, mid_idx, low_idx]
labels = ['high_risk', 'borderline', 'low_risk']

rows = []
for label, idx in zip(labels, sample_indices):
    features = split.X_test.loc[[idx]]
    result = agent_run('Summarise cardiovascular risk for this patient.', features, ml, dl, pp)
    if result.refused or result.explanation is None:
        rows.append({'label': label, 'idx': idx, 'note': 'refused or no explanation'})
        continue
    expl_text = result.explanation['text']
    # Replay the same retrieval the agent used to fetch the chunks
    rag_event = next((e for e in result.events if e.get('event') == 'rag_search'), None)
    query = rag_event['query'] if rag_event else 'cardiovascular risk'
    chunks = search(query, k=4)
    fr = rag_faithfulness(expl_text, chunks)
    rows.append({
        'label': label,
        'idx': idx,
        'n_citations': fr.n_citations,
        'n_unique_citations': fr.n_unique_citations,
        'n_chunks_available': fr.n_chunks_available,
        'invalid_citation_indices': fr.invalid_citation_indices,
        'uncited_chunks_count': fr.uncited_chunks_count,
        'explanation_token_overlap': round(fr.explanation_token_overlap, 3),
    })
faithfulness_df = pd.DataFrame(rows)
faithfulness_df

## 5. Failure-mode summary (for the synthesis paper)

- **Headline metrics** are strong on this small held-out cohort (n~61), but the cohort itself is small enough that confidence intervals would be wide. The headline ROC-AUC alone would oversell the system.
- **Sex slice gap** (visible in both ML and DL above) is data-driven: the UCI Heart Disease set has far fewer female records and a different prevalence. The model card retrieved at explanation time discloses this.
- **Age slice** confirms the model performs better on older bands where positive prevalence is higher, and worse on young patients where the few positive cases are atypical.
- **Top failure cases** show the system is *most confident exactly when it is wrong* on a handful of rows - a calibration smell. Brier score quantifies it. The ensemble + low-confidence flag from Phase E is the structural mitigation: when ML and DL disagree, the orchestrator surfaces "human review recommended" rather than relying on the average alone.
- **RAG faithfulness** above shows zero invalid citations and substantive token overlap with the cited evidence - the structural prompt mitigations from Phase G are holding under live conditions.
- **Hard guardrail**: the refusal substring list short-circuits before any LLM call, so banned requests are not subject to model whim.

**This evaluation is not a clinical validation.** It is the engineering acceptance test for an educational artifact.